# Fink/LSST — Light Curves by Block Selection

This notebook retrieves large photometric light curves (detections + forced photometry)  
for objects found in the **LSST Deep Drilling Fields**.  
Objects are classified using **Fink crossmatch metadata** (`f:xm_*` columns).  
The goal: identify which source category produces the **flattest, most stable flux**  
light curves, suitable for atmospheric transparency calibration.

## Revised strategy

1. **Cone-search** each DDF via `api/v1/conesearch` using **`r:` column prefix**  
   → one alert per detection (columns include `r:diaObjectId`, `r:nDiaSources`, crossmatch flags)
2. **Deduplicate** by `diaObjectId`, keep objects with `nDiaSources >= NP_MIN`
3. **Download full light curves** via `api/v1/sources` + `api/v1/fp`
4. **Classify** each object into a group based on Fink crossmatch metadata (`f:xm_*`)
5. **Compute flatness** σ/⟨f⟩ per group and rank for calibration suitability

## Key discovery about the API

- `/api/v1/conesearch` requires `r:` column prefix (not `i:` — that causes 500 error)
- Block flags (`b_*`) are not filterable in API calls — they are boolean columns in
  `api/v1/sources` but classification is best done via `f:xm_*` crossmatch columns
  present in the conesearch results

```
POST https://api.lsst.fink-portal.org/api/v1/conesearch   → alerts in a sky cone (r: prefix)
POST https://api.lsst.fink-portal.org/api/v1/sources      → full diaSources for one object
POST https://api.lsst.fink-portal.org/api/v1/fp           → forced photometry
GET  https://api.lsst.fink-portal.org/api/v1/blocks       → block flag definitions
```

## Available `f:xm_*` crossmatch columns (confirmed from live API)

| Column | Catalogue | Notes |
|--------|-----------|-------|
| `f:xm_gaiadr3_DR3Name` | Gaia DR3 | source name; `nan` if no match |
| `f:xm_gaiadr3_VarFlag` | Gaia DR3 | `'0'` = not variable; other value = variable |
| `f:xm_gaiadr3_Plx` | Gaia DR3 | parallax in mas (null if no measurement) |
| `f:xm_gaiadr3_e_Plx` | Gaia DR3 | parallax uncertainty in mas |
| `f:xm_simbad_otype` | SIMBAD | object type string (null if no match) |
| `f:xm_legacydr8_pstar` | Legacy DR8 | P(star) photometric classifier [0–1] |
| `f:xm_legacydr8_zphot` | Legacy DR3 | photometric redshift |
| `f:xm_legacydr8_e_zphot` | Legacy DR8 | photo-z uncertainty |
| `f:xm_legacydr8_fqual` | Legacy DR8 | photo-z quality flag |
| `f:xm_mangrove_2MASS_name` | Mangrove | 2MASS name of nearby galaxy |
| `f:xm_mangrove_HyperLEDA_name` | Mangrove/HyperLEDA | HyperLEDA galaxy name |
| `f:xm_mangrove_ang_dist` | Mangrove | angular distance to galaxy (arcsec) |
| `f:xm_mangrove_lum_dist` | Mangrove | luminosity distance to galaxy (Mpc) |
| `f:xm_gcvs_type` | GCVS | General Catalogue of Variable Stars type |
| `f:xm_vsx_Type` | VSX | Variable Star Index type string |
| `f:xm_spicy_class` | SPICY | Young Stellar Object (YSO) classification |
| `f:xm_tns_fullname` | TNS | Transient Name Server name |
| `f:xm_tns_type` | TNS | classified type (e.g. SN Ia) |
| `f:xm_tns_redshift` | TNS | spectroscopic redshift |
| `f:xm_x3hsp_type` | 3HSP | High-Synchrotron-Peaked blazar type |
| `f:xm_x4lac_type` | 4LAC | Fermi-LAT 4th AGN catalogue type |
| `f:clf_cats_class` | Fink ML | multi-class classifier (int) |
| `f:clf_cats_score` | Fink ML | multi-class classifier score [0–1] |
| `f:clf_snnSnVsOthers_score` | Fink ML | SN vs Others score [0–1] |
| `r:extendedness` | LSST | morphology: 0=point-source, 1=extended |
| `r:reliability` | LSST | alert reliability score [0–1] |

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'pandas  version : {pd.__version__}')
print(f'numpy   version : {np.__version__}')

```
| **Name of Field**    | **RA(Degrees)** | **Dec (Degress)**| **Type**          |
| -------------------- | --------------- | ---------------- | ----------------- |
| **Carina**           | 161.5           | -59.7            | Galaxie/Nébuleuse |
| **ELAIS-S1**         | 9.45            | -44.0            | DDF               |
| **COSMOS**           | 150.1           | +2.2             | DDF               |
| **Trifid-Lagoon**    | 270.5           | -23.0            | Nébuleuse         |
| **M49**              | 187.4           | +8.0             | Galaxie           |
| **Rubin_SV_280_-48** | 280.0           | -48.0            | Test SV           |
| **Rubin_SV_320_-15** | 320.0           | -15.0            | Test SV           |
| **Rubin_SV_216_-17** | 216.0           | -17.0            | Test SV           |
| **Rubin_SV_212_-7**  | 212.0           | -7.0             | Test SV           |
| **Rubin_SV_225_-40** | 225.0           | -40.0            | Test SV           |
```

In [ ]:
# ── API base URL ──────────────────────────────────────────────────────────────
FINK_API   = 'https://api.lsst.fink-portal.org'

# ── User parameters ───────────────────────────────────────────────────────────
NP_MIN      = 50       # Minimum nDiaSources per object
NC          = 20       # Number of light curves to plot per group
CONE_RADIUS = 1800.0   # Cone search radius in arcsec (0.5 deg per DDF)
BANDS       = list('ugrizy')

# Gaia parallax SNR threshold for 'gaia_star_stable_hq' (high-quality standard)
# RPlx = Plx / e_Plx >= GAIA_RPLX_MIN  →  reliable geometric distance
GAIA_RPLX_MIN = 5.0

# LSST Deep Drilling Fields (RA/Dec J2000)
DEEP_FIELDS = {
    'COSMOS'   : (150.1191,  2.2058),
    'ELAIS-S1' : (  9.4500, -44.000),
    'XMM-LSS'  : ( 35.7080,  -4.750),
    'ECDFS'    : ( 53.1250, -27.8),
    'EDFS-a'   : (58.9, -49.315),
    'EDFS-b'   : (63.6, -47.6),
    'EDFS'     : (61.24,-48.423),
    'M49'      : (187.4 ,8.0),
}

# ── Classification groups based on Fink crossmatch metadata ──────────────────
# Priority order matters: first match wins.
#
# Groups added in v2 compared to v1:
#   - gaia_star_stable_hq  : Gaia stable + parallax SNR >= GAIA_RPLX_MIN (best calibrators)
#   - vsx_variable         : Variable Star Index match (bad for calibration, flag explicitly)
#   - gcvs_variable        : GCVS match (similar to VSX)
#   - spicy_yso            : Young Stellar Object from SPICY (avoid for calibration)
#   - tns_transient        : Transient Name Server match (SN, AT, etc.)
#   - blazar_x3hsp         : High-Synchrotron-Peaked blazar (3HSP)
#   - blazar_4lac          : Fermi-LAT AGN (4LAC)
#   - mangrove_hyperleda   : HyperLEDA galaxy (companion to 2MASS Mangrove)
#   - legacy_galaxy_photo_z: Legacy galaxy with reliable photo-z (fqual=1)
#   - lsst_extended        : morphologically extended source (r:extendedness ~ 1)
#   - lsst_pointsource     : morphologically compact (r:extendedness ~ 0) without other match

def classify_object(row: dict) -> str:
    """
    Classify an object into a calibration group using Fink crossmatch columns.

    All f:xm_* columns confirmed present in the live Fink/LSST API
    (fink-science v8.39, fink-broker v4.1).

    Returns the group name string.
    """
    # ── Helper: is a string column null/empty? ────────────────────────────────
    def is_null(val):
        return val is None or str(val).strip() in ('', 'None', 'nan', 'Fail', 'null')

    # ── Read relevant columns ─────────────────────────────────────────────────
    simbad   = str(row.get('f:xm_simbad_otype', '')).strip()
    gaia_name= str(row.get('f:xm_gaiadr3_DR3Name', '')).strip()
    gaia_var = str(row.get('f:xm_gaiadr3_VarFlag', '')).strip()
    gaia_plx = row.get('f:xm_gaiadr3_Plx',   None)
    gaia_eplx= row.get('f:xm_gaiadr3_e_Plx', None)
    legacy   = row.get('f:xm_legacydr8_pstar',  None)
    leg_zphot= row.get('f:xm_legacydr8_zphot',  None)
    leg_fqual= row.get('f:xm_legacydr8_fqual',  None)
    mangrove = str(row.get('f:xm_mangrove_2MASS_name',    '')).strip()
    hypleda  = str(row.get('f:xm_mangrove_HyperLEDA_name','') or '').strip()
    gcvs     = str(row.get('f:xm_gcvs_type', '') or '').strip()
    vsx      = str(row.get('f:xm_vsx_Type',  '') or '').strip()
    spicy    = str(row.get('f:xm_spicy_class','') or '').strip()
    tns_name = str(row.get('f:xm_tns_fullname','') or '').strip()
    x3hsp    = str(row.get('f:xm_x3hsp_type', '') or '').strip()
    x4lac    = str(row.get('f:xm_x4lac_type',  '') or '').strip()
    is_sso   = row.get('f:is_sso', False)
    is_cat   = row.get('f:is_cataloged', False)
    extend   = row.get('r:extendedness', None)

    # ── 1. Solar system objects ───────────────────────────────────────────────
    if is_sso:
        return 'solar_system'

    # ── 2. Known transients from TNS (SN, AT, etc.) ───────────────────────────
    # These are variable by definition and should NOT be used for calibration.
    if not is_null(tns_name):
        return 'tns_transient'

    # ── 3. High-energy blazars (3HSP / 4LAC) ─────────────────────────────────
    if not is_null(x3hsp):
        return 'blazar_x3hsp'
    if not is_null(x4lac):
        return 'blazar_4lac'

    # ── 4. Young Stellar Objects from SPICY ──────────────────────────────────
    # YSOs are intrinsically variable in the IR; avoid for flux calibration.
    if not is_null(spicy):
        return 'spicy_yso'

    # ── 5. Variable stars: VSX then GCVS ────────────────────────────────────
    # Flag explicitly before the Gaia branch so that Gaia-matched variables
    # with a VSX/GCVS entry are treated as confirmed variables.
    if not is_null(vsx):
        return 'vsx_variable'
    if not is_null(gcvs):
        return 'gcvs_variable'

    # ── 6. Gaia DR3 stars ────────────────────────────────────────────────────
    if not is_null(gaia_name):
        is_gaia_var = gaia_var not in ('0', '', 'NOT_AVAILABLE', 'None', 'nan', 'Fail')
        if is_gaia_var:
            return 'gaia_star_variable'
        # Non-variable Gaia star: check parallax SNR for high-quality sub-group
        try:
            plx  = float(gaia_plx)  if gaia_plx  is not None else np.nan
            eplx = float(gaia_eplx) if gaia_eplx is not None else np.nan
            rplx = plx / eplx if (np.isfinite(plx) and np.isfinite(eplx) and eplx > 0) else np.nan
        except (TypeError, ValueError):
            rplx = np.nan
        if np.isfinite(rplx) and rplx >= GAIA_RPLX_MIN:
            # Well-measured parallax: confirmed nearby star → best photometric standard
            return 'gaia_star_stable_hq'
        else:
            return 'gaia_star_stable'

    # ── 7. SIMBAD classifications ─────────────────────────────────────────────
    galaxy_types = {
        'G', 'GiG', 'GiC', 'GiP', 'GiR', 'GiS', 'GiPair',
        'Galaxy', 'AGN', 'QSO', 'Sy1', 'Sy2', 'BLL', 'BlL',
        'GinCl', 'GinGroup', 'BClG', 'ClG', 'PairG',
    }
    # Extended list of stellar SIMBAD types (includes explicit variable sub-types)
    star_types = {
        '*', '**', 'SB*', 'PM*', 'V*', 'RGB*', 'LP*', 'Ev*',
        'Em*', 'WR*', 'Be*', 'BS*', 'S*b', 'HB*', 'sg*',
        'Cep', 'RRLyr', 'EB*', 'Mira', 'LPV*', 'SRS', 'RotV*',
        'EllipV*', 'RS*', 'BY*', 'Fl*', 'YSO', 'TTau*', 'HerbObj',
        'pMS*', 'Ae*', 'bCep', 'SX*', 'dS*', 'WVir', 'RV*',
        'deltaCep', 'gammaDor', 'roAp',
    }
    # Explicit variable SIMBAD types (flag for calibration exclusion)
    simbad_variable_types = {
        'Cep', 'RRLyr', 'EB*', 'Mira', 'LPV*', 'SRS', 'RotV*',
        'EllipV*', 'RS*', 'BY*', 'Fl*', 'TTau*', 'bCep', 'SX*',
        'dS*', 'WVir', 'RV*', 'deltaCep', 'gammaDor', 'roAp', 'V*',
        'YSO', 'HerbObj', 'pMS*', 'Ae*',
    }

    if not is_null(simbad):
        if simbad in simbad_variable_types:
            return f'simbad_variable_{simbad}'
        if simbad in star_types:
            return 'simbad_star'
        if simbad in galaxy_types:
            return 'simbad_galaxy'
        # Other SIMBAD type (e.g. nebula, cluster, X-ray source, ...)
        return f'simbad_{simbad}'

    # ── 8. Legacy DR8 photometric star/galaxy separator ───────────────────────
    if legacy is not None and not isinstance(legacy, str):
        try:
            p = float(legacy)
            if p > 0.8:
                return 'legacy_star'
            elif p < 0.2:
                # Sub-classify galaxy: check if photo-z is reliable
                if (leg_fqual is not None and int(leg_fqual) == 1
                        and leg_zphot is not None):
                    return 'legacy_galaxy_photo_z'
                return 'legacy_galaxy'
        except (ValueError, TypeError):
            pass

    # ── 9. Mangrove galaxy catalogues ─────────────────────────────────────────
    if not is_null(mangrove):
        return 'mangrove_galaxy_2mass'
    if not is_null(hypleda):
        return 'mangrove_galaxy_hyperleda'

    # ── 10. LSST morphology fallback ─────────────────────────────────────────
    # Use r:extendedness when no catalogue match is available.
    if extend is not None:
        try:
            ext = float(extend)
            if ext < 0.1:
                return 'lsst_pointsource'
            elif ext > 0.5:
                return 'lsst_extended'
        except (TypeError, ValueError):
            pass

    # ── 11. Fink cataloged but no specific match ──────────────────────────────
    if is_cat:
        return 'fink_cataloged'

    return 'unclassified'


# ── Output directories ────────────────────────────────────────────────────────
NB_TAG   = 'FINK_BLOCK_LC_01'
DIR_DATA = f'data_{NB_TAG}'
DIR_FIGS = f'figs_{NB_TAG}'
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f'Data : {os.path.abspath(DIR_DATA)}')
print(f'Figs : {os.path.abspath(DIR_FIGS)}')

BAND_COLORS = {
    'u': '#9b59b6', 'g': '#2ecc71', 'r': '#e74c3c',
    'i': '#e67e22', 'z': '#3498db', 'y': '#795548',
}
plt.rcParams.update({
    'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 9,
})

def savefig(name):
    for ext in ('pdf', 'png'):
        plt.savefig(os.path.join(DIR_FIGS, f'{name}.{ext}'), bbox_inches='tight')
    print(f'  -> saved {name}.{{pdf,png}}')

print('Configuration done.')

## 2. API wrappers

In [ ]:
def _post_json(url, payload, timeout=90):
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()


def fetch_conesearch(ra: float, dec: float, radius: float,
                     n: int = 1000,
                     columns: str | None = None) -> pd.DataFrame:
    """
    Cone search via /api/v1/conesearch.
    IMPORTANT: column names must use 'r:' or 'f:' prefix — NOT 'i:'.
    Returns one row per alert/diaSource (not per object).
    """
    payload = {
        'ra': ra, 'dec': dec, 'radius': radius,
        'n': n, 'output-format': 'json',
    }
    if columns:
        payload['columns'] = columns
    raw = _post_json(f'{FINK_API}/api/v1/conesearch', payload)
    if not raw:
        return pd.DataFrame()
    return pd.DataFrame(raw)


def fetch_sources(diaObjectId, columns=None) -> pd.DataFrame:
    payload = {'diaObjectId': str(diaObjectId), 'output-format': 'json'}
    if columns:
        payload['columns'] = columns
    raw = _post_json(f'{FINK_API}/api/v1/sources', payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_fp(diaObjectId, columns=None) -> pd.DataFrame:
    payload = {'diaObjectId': str(diaObjectId), 'output-format': 'json'}
    if columns:
        payload['columns'] = columns
    raw = _post_json(f'{FINK_API}/api/v1/fp', payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_objects(diaObjectId, columns=None) -> pd.DataFrame:
    payload = {'diaObjectId': str(diaObjectId), 'output-format': 'json'}
    if columns:
        payload['columns'] = columns
    raw = _post_json(f'{FINK_API}/api/v1/objects', payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


print('API helpers defined.')

## 3. Utility functions

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point

def flux_to_mag(flux_nJy, flux_err_nJy=None):
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid='ignore', divide='ignore'):
        mag = np.where(flux > 0, -2.5*np.log10(flux/AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid='ignore', divide='ignore'):
            mag_err = np.where(flux > 0, 2.5/np.log(10)*np.abs(err/flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f)/np.mean(f)) if len(f) >= 3 else np.nan


def filter_lc(df_lc,
              mjd_col='r:midpointMjdTai', flux_col='r:psfFlux',
              ferr_col='r:psfFluxErr', band_col='r:band', snr_min=3.0):
    df = df_lc.copy()
    for col in (mjd_col, flux_col, ferr_col, band_col):
        if col not in df.columns:
            return pd.DataFrame()
    df[flux_col] = pd.to_numeric(df[flux_col], errors='coerce')
    df[ferr_col] = pd.to_numeric(df[ferr_col], errors='coerce')
    df[mjd_col]  = pd.to_numeric(df[mjd_col],  errors='coerce')
    snr = df[flux_col].abs() / df[ferr_col].replace(0, np.nan)
    df  = df[snr >= snr_min].sort_values(mjd_col).reset_index(drop=True)
    df = df.dropna(subset=[flux_col, ferr_col, mjd_col]).reset_index(drop=True)
    mag, mag_err = flux_to_mag(df[flux_col].values, df[ferr_col].values)
    df['mag']     = mag
    df['mag_err'] = mag_err
    df = df.dropna(subset=['mag', 'mag_err']).reset_index(drop=True)
    return df


print('Utility functions defined.')

## 4. Cone searches on Deep Drilling Fields

**Key fix**: column names in conesearch must use `r:` prefix (not `i:`).  
The `r:nDiaSources` column gives the total detection count per object.

**v2 additions**: the conesearch now also fetches the new crossmatch columns
`f:xm_gaiadr3_Plx`, `f:xm_gaiadr3_e_Plx`, `f:xm_gcvs_type`, `f:xm_vsx_Type`,
`f:xm_spicy_class`, `f:xm_tns_fullname`, `f:xm_x3hsp_type`, `f:xm_x4lac_type`,
`f:xm_mangrove_HyperLEDA_name`, `f:xm_legacydr8_zphot`, `f:xm_legacydr8_fqual`,
and the LSST morphology column `r:extendedness`.

In [ ]:
# Columns to fetch from conesearch — all confirmed from live Fink/LSST API
# NOTE: all use r: or f: prefix — this is the correct prefix for conesearch
COLS_CONE = (
    'r:diaObjectId,r:nDiaSources,r:ra,r:dec,r:band,r:midpointMjdTai,'
    'r:psfFlux,r:psfFluxErr,r:extendedness,'
    # Gaia DR3 (name, variability flag, parallax + uncertainty)
    'f:xm_gaiadr3_DR3Name,f:xm_gaiadr3_VarFlag,'
    'f:xm_gaiadr3_Plx,f:xm_gaiadr3_e_Plx,'
    # SIMBAD
    'f:xm_simbad_otype,'
    # Legacy DR8 (star/galaxy separator + photo-z)
    'f:xm_legacydr8_pstar,f:xm_legacydr8_zphot,f:xm_legacydr8_fqual,'
    # Mangrove (nearby galaxies: 2MASS + HyperLEDA)
    'f:xm_mangrove_2MASS_name,f:xm_mangrove_HyperLEDA_name,'
    # Variable star catalogues (VSX + GCVS)
    'f:xm_vsx_Type,f:xm_gcvs_type,'
    # Young Stellar Objects (SPICY)
    'f:xm_spicy_class,'
    # Transient Name Server
    'f:xm_tns_fullname,f:xm_tns_type,'
    # High-energy blazars (3HSP + 4LAC)
    'f:xm_x3hsp_type,f:xm_x4lac_type,'
    # Fink flags
    'f:is_sso,f:is_cataloged'
)

all_candidates = {}   # diaObjectId (str) → metadata dict

for field_name, (ra, dec) in DEEP_FIELDS.items():
    print(f'\n── Cone search: {field_name}  RA={ra:.4f}  Dec={dec:.4f}  r={CONE_RADIUS:.0f}"')
    try:
        df_cone = fetch_conesearch(ra, dec, CONE_RADIUS, n=2000, columns=COLS_CONE)
    except Exception as e:
        print(f'  ERROR: {e}')
        continue

    if df_cone.empty:
        print('  No alerts returned.')
        continue

    print(f'  Alerts returned: {len(df_cone)}')

    oid_col  = 'r:diaObjectId'
    nsrc_col = 'r:nDiaSources'

    if oid_col not in df_cone.columns:
        print(f'  Column {oid_col!r} missing — cols: {df_cone.columns.tolist()[:10]}')
        continue

    if nsrc_col in df_cone.columns:
        df_cone[nsrc_col] = pd.to_numeric(df_cone[nsrc_col], errors='coerce').fillna(0)

    # Deduplicate: one row per object (keep the one with highest nDiaSources)
    df_obj = (
        df_cone
        .sort_values(nsrc_col, ascending=False)
        .drop_duplicates(subset=[oid_col])
    ) if nsrc_col in df_cone.columns else df_cone.drop_duplicates(subset=[oid_col])

    print(f'  Unique objects: {len(df_obj)}')

    if nsrc_col in df_obj.columns:
        df_ok = df_obj[df_obj[nsrc_col] >= NP_MIN]
    else:
        df_ok = df_obj
    print(f'  Objects with >={NP_MIN} detections: {len(df_ok)}')

    for _, row in df_ok.iterrows():
        oid   = str(row[oid_col])
        nsrc  = int(row.get(nsrc_col, 0))
        group = classify_object(row.to_dict())
        if oid not in all_candidates:
            all_candidates[oid] = {
                'nDiaSources': nsrc,
                'group'      : group,
                'field'      : field_name,
                'ra'         : float(row.get('r:ra', np.nan)),
                'dec'        : float(row.get('r:dec', np.nan)),
            }

    time.sleep(0.5)

print(f'\n=== Total unique candidates: {len(all_candidates)} ===')

import collections
group_counts = collections.Counter(v['group'] for v in all_candidates.values())
print('\nGroup distribution:')
for g, n in sorted(group_counts.items(), key=lambda x: -x[1]):
    print(f'  {g:45s} {n:4d} objects')

## 5. Download light curves for all candidates

In [ ]:
COLS_SRC = 'r:diaObjectId,r:diaSourceId,r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band,r:ra,r:dec,r:snr'
COLS_FP  = 'r:diaObjectId,r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band'

lc_cache = {}  # oid → {'src': df, 'fp': df, 'group': str, 'nDiaSources': int}

MAX_OBJECTS = 200
oids_sorted = sorted(all_candidates, key=lambda o: all_candidates[o]['nDiaSources'], reverse=True)
oids_to_fetch = oids_sorted[:MAX_OBJECTS]

print(f'Objects to fetch: {len(oids_to_fetch)}  (capped at {MAX_OBJECTS})')
print(f'nDiaSources range: {all_candidates[oids_to_fetch[0]]["nDiaSources"]} '
      f'… {all_candidates[oids_to_fetch[-1]]["nDiaSources"]}')

for i, oid in enumerate(oids_to_fetch):
    meta  = all_candidates[oid]
    group = meta['group']
    try:
        df_src = fetch_sources(oid, columns=COLS_SRC)
        df_fp  = fetch_fp(oid,      columns=COLS_FP)
        df_src_f = filter_lc(df_src)
        df_fp_f  = filter_lc(df_fp)
        lc_cache[oid] = {
            'src'         : df_src_f,
            'fp'          : df_fp_f,
            'group'       : group,
            'nDiaSources' : meta['nDiaSources'],
        }
        print(f'  [{i+1:3d}/{len(oids_to_fetch)}] {oid}  '
              f'src={len(df_src_f):4d}  fp={len(df_fp_f):4d}  group={group}')
    except Exception as e:
        print(f'  [{i+1:3d}/{len(oids_to_fetch)}] {oid}  ERROR: {e}')
    time.sleep(0.2)

print(f'\nDownloaded: {len(lc_cache)} objects')

## 6. Build per-group object lists

In [ ]:
all_groups = sorted(set(d['group'] for d in lc_cache.values()))
group_oids = {g: [] for g in all_groups}
for oid, d in lc_cache.items():
    group_oids[d['group']].append(oid)

print('Group → object count:')
for g in sorted(group_oids, key=lambda x: -len(group_oids[x])):
    print(f'  {g:45s} {len(group_oids[g]):3d}')

# Groups excluded from calibration by construction (transients / variables)
CALIBRATION_EXCLUDE = {
    'solar_system', 'tns_transient', 'gaia_star_variable',
    'vsx_variable', 'gcvs_variable', 'spicy_yso',
    'blazar_x3hsp', 'blazar_4lac',
}
# Any group whose name starts with 'simbad_variable' is also excluded
CALIB_GROUPS = [
    g for g in all_groups
    if g not in CALIBRATION_EXCLUDE and not g.startswith('simbad_variable')
]
print(f'\nGroups considered for calibration ({len(CALIB_GROUPS)}):')
for g in CALIB_GROUPS:
    print(f'  {g}')

## 7. Compute flatness metrics per group

In [ ]:
flatness_rows = []

for oid, data in lc_cache.items():
    group = data['group']
    frames = [df for df in (data['fp'], data['src']) if not df.empty]
    if not frames:
        continue
    df_all = pd.concat(frames, ignore_index=True).drop_duplicates(
        subset=['r:midpointMjdTai', 'r:band']
    )
    df_all = df_all.dropna(subset=['r:midpointMjdTai', 'r:psfFlux', 'r:psfFluxErr']).reset_index(drop=True)
    for band in BANDS:
        df_b = df_all[df_all['r:band'] == band]
        if len(df_b) < 5:
            continue
        rms = rms_variability(df_b['r:psfFlux'].values)
        flatness_rows.append({
            'group'        : group,
            'diaObjectId'  : oid,
            'band'         : band,
            'n_pts'        : len(df_b),
            'rms_var'      : rms,
            'mean_flux_nJy': float(df_b['r:psfFlux'].mean()),
        })

df_flat = pd.DataFrame(flatness_rows)

if not df_flat.empty:
    print(f'Flatness table: {len(df_flat)} rows')
    print(df_flat.groupby('group')[['n_pts', 'rms_var']].median().round(4))
else:
    print('No flatness data — check light curve downloads.')

df_flat.to_csv(os.path.join(DIR_DATA, 'flatness_metrics.csv'), index=False)
print('\nSaved flatness_metrics.csv')

## 8. Plot: flatness distribution per group

In [ ]:
if df_flat.empty:
    print('No data.')
else:
    groups_present = [g for g in all_groups if g in df_flat['group'].unique()]
    bands_present  = [b for b in BANDS if b in df_flat['band'].unique()]
    short_labels   = [g.replace('_', '\n') for g in groups_present]

    fig, axes = plt.subplots(1, len(bands_present),
                             figsize=(3.2*len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat['band'] == band]
        data_per_group = [
            df_b[df_b['group'] == g]['rms_var'].dropna().values
            for g in groups_present
        ]
        bp = ax.boxplot(data_per_group, labels=short_labels,
                        patch_artist=True, notch=False, showfliers=True)
        for patch in bp['boxes']:
            patch.set_facecolor(BAND_COLORS.get(band, '#aaa'))
            patch.set_alpha(0.5)
        ax.set_title(f'Band {band}', color=BAND_COLORS.get(band, 'k'), fontweight='bold')
        ax.tick_params(axis='x', rotation=60, labelsize=7)
        ax.set_yscale('log')
        ax.axhline(0.05, ls='--', color='green', lw=0.8, alpha=0.6)

    axes[0].set_ylabel('Normalised RMS  σ/<f>')
    fig.suptitle('Flux variability by source class — lower = flatter',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    savefig('01_flatness_boxplot_by_group')
    plt.show()

## 9. Plot light curves — top-NC objects per group

Only groups in `CALIB_GROUPS` are plotted (transients and confirmed variables excluded).

In [ ]:
def rank_oids(oid_list, nc=NC):
    scored = [(o, len(lc_cache[o]['fp']) + len(lc_cache[o]['src'])) for o in oid_list if o in lc_cache]
    return [o for o, _ in sorted(scored, key=lambda x: -x[1])[:nc]]


def plot_lc_grid(oid_list, group, mode='flux', nc=NC):
    top = rank_oids(oid_list, nc)
    n_obj = len(top)
    if n_obj == 0:
        print(f'  No objects for group {group}.')
        return
    fig, axes = plt.subplots(n_obj, len(BANDS),
                             figsize=(2.8*len(BANDS), 2.6*n_obj),
                             sharex=False, sharey=False, squeeze=False)

    for row_idx, oid in enumerate(top):
        d = lc_cache[oid]
        frames = [df for df in (d.get('fp', pd.DataFrame()), d.get('src', pd.DataFrame())) if not df.empty]
        if not frames:
            continue
        df_all = pd.concat(frames, ignore_index=True).drop_duplicates(
            subset=['r:midpointMjdTai', 'r:band'])
        df_all = df_all.dropna(subset=['r:midpointMjdTai', 'r:psfFlux', 'r:psfFluxErr']).reset_index(drop=True)
        for col_idx, band in enumerate(BANDS):
            ax   = axes[row_idx][col_idx]
            df_b = df_all[df_all['r:band'] == band].sort_values('r:midpointMjdTai')
            if len(df_b) < 3:
                ax.set_visible(False)
                continue
            if mode == 'flux':
                mask = np.isfinite(df_b['r:psfFlux'].values) & np.isfinite(df_b['r:psfFluxErr'].values)
            else:
                mask = np.isfinite(df_b['mag'].values) & np.isfinite(df_b['mag_err'].values)
            df_b = df_b[mask].reset_index(drop=True)
            if len(df_b) < 3:
                ax.set_visible(False)
                continue
            dt    = df_b['r:midpointMjdTai'].values
            dt = dt - dt.min()
            color = BAND_COLORS.get(band, 'gray')
            if mode == 'flux':
                y, yerr = df_b['r:psfFlux'].values, df_b['r:psfFluxErr'].values
            else:
                y, yerr = df_b['mag'].values, df_b['mag_err'].values
                ax.invert_yaxis()
            ax.errorbar(dt, y, yerr=yerr, fmt='o', ms=3, lw=0.8, elinewidth=0.8,
                        color=color, alpha=0.8)
            rms = rms_variability(df_b['r:psfFlux'].values)
            ax.set_title(f'{band} N={len(df_b)} rms={rms:.3f}',
                         fontsize=7, pad=2, color=color)
            ax.set_xlabel('Δt (days)', fontsize=7)
            ax.tick_params(labelsize=7)

        axes[row_idx][0].set_ylabel(f'{oid}\n{"flux (nJy)" if mode=="flux" else "AB mag"}',
                                    fontsize=10)

    fig.suptitle(f'Group: {group}  |  mode={mode}', fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    safe = group.replace('/', '_').replace(' ', '_')
    savefig(f'02_lc_{safe}_{mode}')
    plt.show()


print('Plot functions defined.')

In [ ]:
# Plot calibration-candidate groups only
groups_to_plot = [g for g in CALIB_GROUPS if len(group_oids.get(g, [])) >= 2]

for group in groups_to_plot:
    print(f'\n=== {group} ({len(group_oids[group])} objects) ===')
    plot_lc_grid(group_oids[group], group, mode='flux')

In [ ]:
for group in groups_to_plot:
    print(f'\n=== {group} (magnitude) ===')
    plot_lc_grid(group_oids[group], group, mode='mag')

## 10. Calibration summary scatter plot

In [ ]:
if df_flat.empty:
    print('No data.')
else:
    summary = (
        df_flat
        .groupby(['group', 'band'])
        .agg(median_rms=('rms_var','median'), mean_flux=('mean_flux_nJy','mean'),
             n_obj=('diaObjectId','nunique'), n_pts=('n_pts','sum'))
        .reset_index()
    )
    bands_present = [b for b in BANDS if b in summary['band'].unique()]
    fig, axes = plt.subplots(1, len(bands_present), figsize=(4.5*len(bands_present), 5))
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = summary[summary['band'] == band]
        ax.scatter(df_b['mean_flux'], df_b['median_rms'],
                   s=80*np.sqrt(df_b['n_pts']/max(df_b['n_pts'].max(), 1)+0.1),
                   c=BAND_COLORS.get(band, 'gray'), alpha=0.8, edgecolors='k', linewidths=0.5)
        for _, row in df_b.iterrows():
            ax.annotate(row['group'], (row['mean_flux'], row['median_rms']),
                        fontsize=6, ha='left', va='bottom',
                        xytext=(3,2), textcoords='offset points')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('Mean flux (nJy)')
        ax.set_ylabel('Median σ/<f>')
        ax.set_title(f'Band {band}', color=BAND_COLORS.get(band,'k'), fontweight='bold')
        ax.axhline(0.05, ls='--', color='green', lw=1, alpha=0.6, label='5%')
        ax.legend(fontsize=7)

    fig.suptitle('Calibration suitability  (best = bright & flat = bottom-right)',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    savefig('03_calibration_summary')
    plt.show()

    print('\nFull summary table:')
    display(summary.sort_values(['band', 'median_rms']).head(40))

## 11. Save all light curves to Parquet

In [ ]:
for group in all_groups:
    oids = group_oids[group]
    all_fp, all_src = [], []
    for oid in oids:
        d = lc_cache.get(oid, {})
        for df, store in ((d.get('fp', pd.DataFrame()), all_fp),
                           (d.get('src', pd.DataFrame()), all_src)):
            if not df.empty:
                tmp = df.copy()
                tmp['diaObjectId'] = oid
                tmp['group']       = group
                store.append(tmp)
    safe = group.replace('/', '_')
    for store, tag in ((all_fp, 'fp'), (all_src, 'src')):
        if store:
            path = os.path.join(DIR_DATA, f'{safe}_{tag}.parquet')
            pd.concat(store, ignore_index=True).to_parquet(path, index=False)
            print(f'  Saved {path}')

print('\nAll data saved.')

## 12. Final ranking — recommended source class for calibration

In [ ]:
if df_flat.empty:
    print('No flatness data.')
else:
    # Restrict ranking to calibration-candidate groups
    df_calib = df_flat[df_flat['group'].isin(CALIB_GROUPS)]
    ranking = (
        df_calib
        .groupby('group')
        .agg(n_objects=('diaObjectId','nunique'), n_points=('n_pts','sum'),
             median_rms=('rms_var','median'), mean_flux_nJy=('mean_flux_nJy','mean'))
        .sort_values('median_rms')
        .reset_index()
    )
    ranking['mean_mag'] = -2.5 * np.log10(ranking['mean_flux_nJy'] / AB_FLUX_ZERO)

    print('Ranking by photometric flatness (ascending RMS) — calibration groups only:')
    print('='*90)
    print(ranking[['group','n_objects','n_points','median_rms','mean_mag']]
          .to_string(index=False, float_format='{:.4f}'.format))

    best = ranking.iloc[0]
    print(f'\n==> Best class for atmospheric transparency calibration:')
    print(f'    {best["group"]}  (median σ/<f> = {best["median_rms"]:.4f},  mean mag ≈ {best["mean_mag"]:.2f})')
    print(f'\nNote: gaia_star_stable_hq (Plx/ePlx >= {GAIA_RPLX_MIN}) is the purest '
          f'geometric-parallax subsample if available.')